  # Large-Scale Thematic Search with Smart Batching







  This notebook shows how to search for thematic information across a large universe of companies, process the results, and build sentiment signals. The workflow uses Smart Batching for efficient API usage and LLM validation for high-precision filtering.







  **Workflow overview:**



  1. **Theme Decomposition**: break down a main theme into sub-themes for higher recall



  2. **Search Planning & Execution**: create and execute optimized search plans



  3. **Masking & LLM Validation**: mask company names to avoid bias, validate theme relevance and impact



  4. **Signal Construction**: build rolling sentiment signals and visualize top companies







  **Create a `.env` file with:**

  **WorldQuant Brain (required):**
  ```BRAIN_EMAIL=your_email```
  ```BRAIN_PASSWORD=your_password```

  **Optional:** `OPENAI_API_KEY` (for the LLM validation step).

  **Note:** Restart the kernel if you change API credentials, as they are read at import time.

 ## Imports

In [ ]:
import asyncio
import json
import logging
import os
import pickle
import sys
from pathlib import Path

import nest_asyncio
import pandas as pd
import plotly.io as pio
from dotenv import load_dotenv
load_dotenv()

import sys
# Import all from src module
from src import (
    convert_to_dataframe,
    deduplicate_documents,
    execute_search,
    execute_full_grid_search,
    explode_to_dataframe,
    load_plan,
    plan_search,
    save_plan,
    aggregate_results_by_chunk,
    extract_companies_from_entity_list,
    get_unknown_entities_from_df_column,
    keep_only_companies_in_detections,
    map_create_only_companies_column,
)

from src.helper import (
    build_rolling_impact_signal,
    plot_top_entities_rolling_signal,
    mask_companies_in_df,
)

from src.labeler.screener_labeler import Labeler, merge_validation_labels

from src.mindmap import generate_risk_tree, generate_theme_tree

from session import BrainSession
# WorldQuant Brain API Configuration
API_BASE_URL = "https://api.worldquantbrain.com"
EMAIL = os.getenv("BRAIN_EMAIL")
PASSWORD = os.getenv("BRAIN_PASSWORD")
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

session = BrainSession(
    base_url=API_BASE_URL,
    email=EMAIL,
    password=PASSWORD
)


 ## Configuration

In [ ]:
# Suppress verbose HTTP logging
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)

# Add current directory to path for local imports
sys.path.insert(0, str(Path.cwd()))

# Apply nest_asyncio if needed (for Jupyter environments)
try:
    asyncio.get_running_loop()
    nest_asyncio.apply()
    print("✅ nest_asyncio applied")
except RuntimeError:
    pass  # No running loop, nest_asyncio not needed

# Configure Plotly renderer
try:
    if 'JUPYTERHUB_SERVICE_PREFIX' in os.environ or 'JPY_SESSION_NAME' in os.environ:
        pio.renderers.default = 'jupyterlab'
    else:
        pio.renderers.default = 'plotly_mimetype+notebook'
except Exception:
    pio.renderers.default = 'notebook'



  ## Setup & Authentication

In [ ]:
# Load credentials from .env file
import os
from dotenv import load_dotenv
load_dotenv()

# WorldQuant Brain API Configuration
API_BASE_URL = "https://api.worldquantbrain.com"
EMAIL = os.getenv("BRAIN_EMAIL")
PASSWORD = os.getenv("BRAIN_PASSWORD")
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

session = BrainSession(
    base_url=API_BASE_URL,
    email=EMAIL,
    password=PASSWORD
)

  ## Search Parameters

In [ ]:
# Main theme to analyze - will be decomposed into sub-themes for higher recall
theme = "Semiconductor supply shortage disrupted global production"

# Universe: CSV file with columns 'id' (entity ID) and 'name' (company name)
universe_csv = "id_name_mapping_us_top_3000_small.csv"

# Time window for the search
start_date = "2021-01-01"
end_date = "2021-06-30"

# Sampling percentage: controls the % of chunks to retrieve per basket (0.0-1.0)
# Lower values = faster execution, higher values = more comprehensive results
chunk_percentage = 0.5

  ## 1. Theme Decomposition







  Two functions are available to decompose a main theme into sub-themes using an LLM:



  - **`generate_risk_tree`**: Decomposes the theme into risk/impact categories (e.g., costs, supply chain disruption, revenue decline). Best for risk-related themes.



  - **`generate_theme_tree`**: Decomposes the theme into general sub-topics without a risk focus. Best for broader thematic exploration.







  Both functions generate a tree structure where terminal nodes contain specific, searchable summaries. Each sub-theme becomes a separate search query, and results are merged and deduplicated. This increases recall by capturing different aspects of the main theme that might be expressed differently in news articles. Higher recall means fewer missed events, so your final signal is more complete.







In [ ]:
# Generate risk tree from main theme using LLM
# Alternative: use generate_theme_tree(main_theme=theme) for general thematic decomposition
risk_tree = generate_risk_tree(
    main_theme=theme
)

# Extract terminal nodes: these are the specific sub-themes we'll search for
terminal_labels_and_summaries = risk_tree.get_terminal_label_summaries()

In [ ]:
terminal_labels_and_summaries

In [ ]:
# For faster testing, only use the first item from terminal_labels_and_summaries
terminal_labels_and_summaries = {'Production Delays':'Companies will face production delays due to semiconductor supply shortages disrupting global production.'}

  ## 2. Search Planning with Smart Batching







  For each sub-theme, we create a search plan using `plan_search`. Smart Batching optimizes large-scale searches by grouping entities into baskets based on estimated co-mention volume, splitting time periods to balance API load, and estimating chunk counts for proportional sampling.







  **Key parameters:**



  - **`volume_query_mode="iterative"`**: Iteratively refines volume estimates for better basket sizing.



  - **`reranker_enabled=True` + `reranker_threshold=0.7`**: Enables semantic reranking and filters chunks below 0.7 similarity, reducing noise and LLM post-processing costs.



  - **`volume_correction=(0.7, 400)`**: The API used to retrieve the volume doesn't support similarity thresholds, so it overestimates chunk counts. This applies a correction factor (0.7) with a minimum (400) to prevent too many splitted high baskets.



  - **`min_period_days=30`**: Minimum time window size for each basket.

In [ ]:
# Create search plans for each sub-theme
plans_list = []
for i, summary in enumerate(terminal_labels_and_summaries.values()):
    plan = plan_search(
        text=summary,
        universe_csv_path=universe_csv,
        start_date=start_date,
        end_date=end_date,
        session=session,
        volume_query_mode="iterative",
        max_iterations_per_batch=10,
        reranker_enabled=True,
        reranker_threshold=0.5,  
        min_period_days=30,  # Minimum time window size for each basket
    )

    
    # Save plan to file for reproducibility and debugging
    plan_file = f"search_plan_node_{i}.json"
    plans_list.append(plan_file)
    save_plan(plan, plan_file)

    print(f"Theme {i}: {list(terminal_labels_and_summaries.keys())[i]}")
    print(f"  Total expected chunks: {plan['total_expected_chunks']:,}")
    print(f"  Number of baskets: {len(plan['baskets'])}")

  ## 3. Search Execution







  Execute the search plans with parallel processing and rate limiting. The `execute_search` function:



  - **Parallel processing**: Processes baskets concurrently using `max_workers` threads



  - **Rate limiting**: Respects API limits via `requests_per_minute`



  - **Proportional sampling**: Applies `chunk_percentage` to sample from estimated volumes









  Results are deduplicated to merge chunks from the same document that may appear in multiple baskets.

In [ ]:
# Execute searches for all sub-themes
results_list = []
for plan_name in plans_list:
    plan = load_plan(plan_name)
    
    results_raw = execute_search(
        search_plan=plan,
        chunk_percentage=chunk_percentage,  # Sample this percentage of estimated chunks
        requests_per_minute=100,  # API rate limit
        session=session,
    )

    # Deduplicate: merge chunks from same document across different baskets
    results = deduplicate_documents(results_raw)
    
    # Save raw results for debugging/reproducibility
    with open(f'results_{plan_name}', 'w') as f:
        json.dump(results, f)
    results_list.append(results)

    print(f"\n Search complete for {plan_name}!")
    print(f"   Retrieved {len(results):,} deduplicated documents")

## 3.1 Standard Full Grid Search (No Smart Batching)

This is the standard approach for performing large-scale searches. It queries a full grid of entities using the same parameters (text query, date range, entity filter).

Advantages:
- No need to query the co-mention endpoint to estimate volume for basket creation
- More efficient when preserving distribution across companies is not an issue (e.g., small time windows with limited data volumes or niche topics)

Disadvantages:

While you can add multiple companies to the any_of filter, this risks having high media-attention companies dominate the results and bury those with lower coverage. Since each call is limited to 1,000 chunks, companies with very high chunk volumes in that time window can consume most or all of the quota, leaving other companies in the same batch with few or no chunks. This results in uneven and potentially incomplete data retrieval where some entities are over-represented while others are under-represented or missing entirely.The smart batching approach addresses this issue by distributing queries more evenly, but it requires a search plan leveraging the comention endpoints. 





In [ ]:
# Standard Full Grid Search

#fg_search_batch_size = 500  # number of companies per API request (API maximum)
#fg_search_max_chunks_per_request = 1000  # max chunks per query (API limit)
#fg_search_requests_per_minute = 50

#results_list = []
#for i, summary in enumerate(terminal_labels_and_summaries.values()):
#    results_raw = execute_full_grid_search(
#        text=summary,
#        universe_csv_path=universe_csv,
#        start_date=start_date,
#        end_date=end_date,
#        batch_size=fg_search_batch_size,
#        session=session,
#        requests_per_minute=fg_search_requests_per_minute,
#        max_chunks_per_request=fg_search_max_chunks_per_request,
#    )

    # Deduplicate: merge chunks from same document across different batches
#    results = deduplicate_documents(results_raw)

#    results_list.append(results)
#    label = list(terminal_labels_and_summaries.keys())[i]
#    print(f"\n Normal search complete for theme {i}: {label}")
#    print(f"   Retrieved {len(results):,} deduplicated documents")

  ## 4. Results Aggregation and Labeling







  Combine results from all sub-theme searches into a single DataFrame:



  - **Tagging**: Each chunk is tagged with `label` (the sub-theme label) and `theme` (the full description)



  - **Deduplication**: Chunks appearing in multiple searches are merged, with labels/themes aggregated into lists

In [ ]:
# Convert results to DataFrames and tag with sub-theme labels
df_results_list = []
labels_summaries = list(terminal_labels_and_summaries.items())

for results, (label, theme_text) in zip(results_list, labels_summaries):
    df_exploded_by_chunk = convert_to_dataframe(results)
    df_exploded_by_chunk['label'] = label
    df_exploded_by_chunk['theme'] = theme_text
    df_results_list.append(df_exploded_by_chunk)

# Concatenate all results
df_concat = pd.concat(df_results_list, ignore_index=True)

# Aggregate by chunk (chunk_text): same chunk from different themes gets label/theme as lists
df_exploded_by_chunk = aggregate_results_by_chunk(df_concat)

print(f"Total unique chunks: {len(df_exploded_by_chunk):,}")

In [ ]:
# Save checkpoint (uncomment to save - useful for long-running workflows)
# with open('df_exploded_by_chunk.pkl', 'wb') as f:
#     pickle.dump(df_exploded_by_chunk, f)

In [ ]:
# Load from checkpoint (uncomment to skip search if already completed)
# with open('df_exploded_by_chunk.pkl', 'rb') as f:
#     df_exploded_by_chunk = pickle.load(f)

  ## 5. Entity Resolution and Company Filtering







  We keep only companies in our universe CSV so the pipeline outputs one row per (company, date) and the signal is defined only for our investable universe. Search results contain entity IDs from the BigData knowledge graph; some entities may not be in our target universe (the input CSV). This step:



  1. **Identify unknown entities**: Find entity IDs not in our universe CSV



  2. **Resolve via API**: Query the Entities API (`/v1/knowledge-graph/entities/id`) to get entity metadata



  3. **Filter to companies only**: Keep only entities with `category='companies'`







  Multiple API calls are made to resolve unknown entity IDs in batches.

In [ ]:
# Resolve unknown entities and filter to companies only
# This makes API calls to the Knowledge Graph to identify which entities are companies
entities = get_unknown_entities_from_df_column(
    df_exploded_by_chunk, 
    "entity_ids", 
    universe_csv
)

# Query API to get company information for unknown entities
other_companies = extract_companies_from_entity_list(entities, session)

# Create column with only company entity IDs
df_exploded_by_chunk_only_companies = map_create_only_companies_column(
    df_exploded_by_chunk, 
    universe_csv, 
    other_companies
)

# Filter detections to keep only companies
df_detection = keep_only_companies_in_detections(df_exploded_by_chunk_only_companies)

  ### 5.1 Explode by Entity







  Transform from chunk-level to entity-level DataFrame. Each row represents one (chunk, entity) pair, filtered to include only entities from our target universe.

In [ ]:
# Explode: one row per (chunk, entity) pair, filtered by universe
df_exploded_by_entity = explode_to_dataframe(
    df_detection,
    universe_csv=universe_csv
)

print(f"Exploded DataFrame shape: {df_exploded_by_entity.shape}")
print(f"Unique entities: {df_exploded_by_entity['entity_id'].nunique()}")
df_exploded_by_entity.head()

  ## 6. Company Masking (Optional but Recommended)







  Before LLM validation, we mask company names in the text to avoid bias. LLMs may have pre-existing associations with well-known companies that could influence their judgment.







  **How masking works:**



  - Target company (the one we're analyzing) → `[TARGET_COMPANY]`



  - Other companies mentioned → `[OTHER_COMPANY]`







  This ensures the LLM evaluates the content objectively based on the text, not company reputation. So the impact label is less biased by company name.

In [ ]:
# Mask company names to avoid LLM bias
df_masked = mask_companies_in_df(df_exploded_by_entity)
print(f"Masked DataFrame shape: {df_masked.shape}")

  ## 7. LLM Validation: Theme Relevance and Impact Assessment







  Use an LLM to validate each (chunk, entity) pair. Two outputs are produced:



  1. **Theme Relevance (`is_theme_related`)**: Is this chunk actually about our theme? Filters out false positives from semantic search.



  2. **Impact Assessment (`impact`)**: How does the theme affect this company?



     - `+1`: Positive impact (company benefits)



     - `-1`: Negative impact (company is harmed)



     - `0`: Neutral or unclear impact







  We aggregate these into a daily score per company and then smooth with rolling windows to get a time series you can use for ranking or backtesting.



  The reranker threshold in search planning already filters low-relevance chunks, reducing the volume of LLM calls needed here.







  **Note:** The validation prompts in `src/labeler` are customizable. Modify them to adjust the criteria for theme relevance or impact assessment based on your needs.

In [ ]:
# Initialize labeler with LLM configuration
labeler = Labeler(llm_model_config="openai::gpt-4o-mini")


# Run LLM validation in parallel
labels_df = labeler.get_validation_labels(
    main_theme=theme,
    df_masked=df_masked,
    timeout=20,
    max_workers=10,  # Parallel LLM calls
)

# Merge labels back to original DataFrame
df_with_labels = merge_validation_labels(df_masked, labels_df)

print(f"Labeled DataFrame shape: {df_with_labels.shape}")
print(f"Theme-related chunks: {df_with_labels['is_theme_related'].sum():,}")
df_with_labels.head()



In [ ]:
# Save checkpoint (uncomment to save)
with open('df_with_labels.pkl', 'wb') as f:
    pickle.dump(df_with_labels, f)



In [ ]:
# Load from checkpoint (uncomment to skip LLM validation if already completed)
# with open('df_with_labels.pkl', 'rb') as f:
#     df_with_labels = pickle.load(f)



  ## 8. Rolling Impact Signal Construction







  Build a daily sentiment signal for each company based on LLM impact labels. The signal is **net positive vs negative theme impact over the window**, per company. Use **7-day** for a more responsive signal and **30-day** for a smoother, more stable one. Rank companies by `signal_30d` to see who is most affected by the theme; use as a long/short input or as a feature in a model.



  - **Filtering**: Only chunks where `is_theme_related=True` are included



  - **Daily aggregation**: Impact scores (+1/-1/0) are averaged per company per day



  - **Smoothing**: Backward-looking rolling windows (7-day and 30-day) smooth the signal









In [ ]:
# Build rolling impact signal from LLM labels
df_rolling_signal = build_rolling_impact_signal(
    df_with_labels,
    entity_col="entity_name",
    date_col="date",
    impact_col="impact",
    is_theme_related_col="is_theme_related",
    window_7d=7,
    window_30d=30,
    rolling_agg="sum",  # "mean" = media rolling, "sum" = somma rolling
)
df_rolling_signal



  ### 8.1 Visualize Top Companies by Volume







  Plot the rolling 30-day impact signal for the top 5 companies by news volume.

In [ ]:
# Plot rolling signal for top 5 companies by volume
plot_top_entities_rolling_signal(
    df_rolling_signal,
    entity_col="entity_name",
    date_col="date",
    signal_col="signal_30d",  # with the relative padding from window
    top_n=5,
)



In [ ]:
# Save DF
df_rolling_signal.to_csv("df_rolling_signal.csv", index=False)


  ## Summary







  This workflow showed a complete pipeline: theme decomposition for higher recall, Smart Batching for efficient large-scale search, entity resolution via Knowledge Graph API, LLM validation for theme relevance and impact assessment, and rolling signal construction for trend analysis. The main output is a **signal table** (e.g. `df_rolling_signal`): one row per (entity, date) with `signal_7d` and `signal_30d`, ready for backtesting or screening.



